# Why Adaptive Stopping? Introduction to Empirical Bernstein Stopping

This notebook motivates the need for **Empirical Bernstein Stopping (EBS)** over fixed Hoeffding planning by showing how variance-aware methods can significantly reduce shot requirements.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qamp_shotplanner.planners.hoeffding import HoeffdingPlanner
from qamp_shotplanner.planners.ebs_stopping import EmpiricalBernsteinStopper

# For reproducibility
np.random.seed(42)

## 1. The Problem with Hoeffding: Worst-Case Assumptions

Hoeffding's inequality provides a **data-independent** bound for estimating the mean of a bounded random variable $X \in [a, b]$:

$$
n = \frac{R^2}{2\epsilon^2} \ln\left(\frac{2}{\delta}\right)
$$

where $R = b - a$ is the **full range** of the variable.

### Key Issue: Worst-Case Variance

Hoeffding assumes the worst-case variance:
$$
\sigma^2_{\text{worst}} = \frac{R^2}{4}
$$

This occurs when the distribution is maximally spread (e.g., 50-50 split for binary outcomes). However, many real quantum systems have **much lower variance**, leading to wasted shots.

## 2. Variance Matters: Two Coin Examples

Let's consider two coins, both giving outcomes in $\{0, 1\}$ (so $R = 1$), but with different variances:

1. **Biased coin**: $p = 0.9$ (heads with 90% probability)
   - Mean: $\mu = 0.9$
   - Variance: $\sigma^2 = p(1-p) = 0.09$ (low variance)

2. **Fair coin**: $p = 0.5$ (heads with 50% probability)
   - Mean: $\mu = 0.5$
   - Variance: $\sigma^2 = 0.25$ (worst-case variance)

In [ ]:
# Define the two scenarios
p_biased = 0.9  # Low variance
p_fair = 0.5    # Worst-case variance

# Compute theoretical variances
var_biased = p_biased * (1 - p_biased)
var_fair = p_fair * (1 - p_fair)

print("=== Two Coin Scenarios ===")
print(f"\nBiased coin (p={p_biased}):")
print(f"  Mean: {p_biased}")
print(f"  Variance: {var_biased:.4f}")
print(f"  Std dev: {np.sqrt(var_biased):.4f}")

print(f"\nFair coin (p={p_fair}):")
print(f"  Mean: {p_fair}")
print(f"  Variance: {var_fair:.4f}")
print(f"  Std dev: {np.sqrt(var_fair):.4f}")

print(f"\nVariance ratio: {var_fair / var_biased:.2f}x")
print("Hoeffding uses worst-case variance for BOTH scenarios!")

## 3. Hoeffding Planning: Same Shots for Both

Let's plan shots using Hoeffding for both scenarios with $\epsilon = 0.05$ and $\delta = 0.01$:

In [ ]:
epsilon = 0.05
delta = 0.01
a, b = 0.0, 1.0  # Coin outcomes

# Hoeffding planner (same for both scenarios)
hoeffding = HoeffdingPlanner(epsilon_stat=epsilon, delta=delta, a=a, b=b)
hoeffding_shots = hoeffding.planned_shots()

print(f"Hoeffding planning:")
print(f"  Tolerance: ε = {epsilon}")
print(f"  Confidence: 1 - δ = {1-delta:.2%}")
print(f"  Planned shots: {hoeffding_shots:,}")
print(f"\nBoth scenarios use the SAME number of shots!")
print(f"This is wasteful for the low-variance biased coin.")

## 4. Enter Empirical Bernstein Stopping (EBS)

**EBS** adapts to the observed variance during sampling:

$$
\epsilon_n = \bar{\sigma}_n \sqrt{\frac{2\ln(3/\delta)}{n}} + R \frac{3\ln(3/\delta)}{n}
$$

where $\bar{\sigma}_n^2$ is the **empirical variance** computed from the samples.

### Key Advantages:

1. **Variance-aware**: Uses actual observed variance $\bar{\sigma}_n^2$ instead of worst-case $R^2/4$
2. **Adaptive stopping**: Stops early when confidence threshold is met
3. **Safety guarantee**: Never exceeds Hoeffding's cap (worst-case protection)

## 5. EBS in Action: Low vs High Variance

Let's run EBS on both coin scenarios and see how many shots it actually uses:

In [ ]:
# Create EBS stoppers for both scenarios
ebs_biased = EmpiricalBernsteinStopper(
    epsilon_stat=epsilon,
    delta=delta,
    a=a,
    b=b,
    beta=1.1,
    alpha=1.0,
    n_min=10,
)

ebs_fair = EmpiricalBernsteinStopper(
    epsilon_stat=epsilon,
    delta=delta,
    a=a,
    b=b,
    beta=1.1,
    alpha=1.0,
    n_min=10,
)

print(f"EBS configuration:")
print(f"  Max shots (Hoeffding cap): {ebs_biased.planned_max_shots():,}")
print(f"  Beta (checkpoint growth): {ebs_biased.beta}")
print(f"  Alpha (tightness): {ebs_biased.alpha}")
print(f"  Checkpoints: {len(ebs_biased.checkpoints())} total")

In [ ]:
# Run EBS on biased coin
def sample_biased():
    return float(np.random.rand() < p_biased)

result_biased = ebs_biased.run(sample_biased)

print("=== EBS Result: Biased Coin (low variance) ===")
print(f"Estimate: {result_biased.estimate:.4f} (true: {p_biased})")
print(f"Shots used: {result_biased.n:,} / {hoeffding_shots:,}")
print(f"Stopped by: {result_biased.stopped_by}")
print(f"Final radius: {result_biased.epsilon_n:.6f}")
print(f"Target radius: {epsilon:.6f}")
print(f"Observed variance: {result_biased.stats.variance_biased:.6f}")
print(f"\nShot savings: {hoeffding_shots - result_biased.n:,} ({100*(1 - result_biased.n/hoeffding_shots):.1f}%)")

In [ ]:
# Run EBS on fair coin
def sample_fair():
    return float(np.random.rand() < p_fair)

result_fair = ebs_fair.run(sample_fair)

print("=== EBS Result: Fair Coin (worst-case variance) ===")
print(f"Estimate: {result_fair.estimate:.4f} (true: {p_fair})")
print(f"Shots used: {result_fair.n:,} / {hoeffding_shots:,}")
print(f"Stopped by: {result_fair.stopped_by}")
print(f"Final radius: {result_fair.epsilon_n:.6f}")
print(f"Target radius: {epsilon:.6f}")
print(f"Observed variance: {result_fair.stats.variance_biased:.6f}")

if result_fair.stopped_by == 'cap':
    print(f"\nEBS hit the Hoeffding cap (as expected for high variance)")
else:
    print(f"\nShot savings: {hoeffding_shots - result_fair.n:,} ({100*(1 - result_fair.n/hoeffding_shots):.1f}%)")

## 6. Visualization: Shots Used vs Variance

Let's sweep across different coin biases to see how EBS adapts:

In [ ]:
# Sweep over different coin biases
biases = np.linspace(0.1, 0.9, 17)
variances = biases * (1 - biases)
ebs_shots_list = []

for p in biases:
    # Create fresh EBS stopper
    ebs = EmpiricalBernsteinStopper(
        epsilon_stat=epsilon,
        delta=delta,
        a=a,
        b=b,
        beta=1.1,
        alpha=1.0,
        n_min=10,
    )
    
    # Sample from this coin
    def sample():
        return float(np.random.rand() < p)
    
    result = ebs.run(sample)
    ebs_shots_list.append(result.n)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(variances, ebs_shots_list, 'o-', label='EBS (adaptive)', linewidth=2, markersize=6)
ax.axhline(hoeffding_shots, color='red', linestyle='--', linewidth=2, label='Hoeffding (fixed)')

ax.set_xlabel('True Variance σ²', fontsize=12)
ax.set_ylabel('Shots Used', fontsize=12)
ax.set_title('EBS Adapts to Variance, Hoeffding Does Not', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Mark the worst-case variance
ax.axvline(0.25, color='red', linestyle=':', alpha=0.5, linewidth=1.5)
ax.text(0.25, hoeffding_shots * 0.5, 'Worst-case\nvariance', 
        ha='center', va='center', fontsize=10, color='red')

plt.tight_layout()
plt.show()

print(f"\nKey observation:")
print(f"  - Low variance (< 0.25): EBS uses fewer shots")
print(f"  - Worst-case variance (= 0.25): EBS hits the Hoeffding cap")
print(f"  - EBS provides best-case speedup, worst-case safety")

## 7. Comparison Summary

Let's create a comparison table:

In [ ]:
import pandas as pd

# Re-run both scenarios for clean results
np.random.seed(42)

ebs_biased_2 = EmpiricalBernsteinStopper(epsilon_stat=epsilon, delta=delta, a=a, b=b, beta=1.1, alpha=1.0, n_min=10)
ebs_fair_2 = EmpiricalBernsteinStopper(epsilon_stat=epsilon, delta=delta, a=a, b=b, beta=1.1, alpha=1.0, n_min=10)

res_biased = ebs_biased_2.run(lambda: float(np.random.rand() < p_biased))
res_fair = ebs_fair_2.run(lambda: float(np.random.rand() < p_fair))

comparison = pd.DataFrame([
    {
        'Scenario': 'Biased coin (p=0.9)',
        'True Variance': var_biased,
        'Hoeffding Shots': hoeffding_shots,
        'EBS Shots': res_biased.n,
        'Savings': hoeffding_shots - res_biased.n,
        'Savings %': 100 * (1 - res_biased.n / hoeffding_shots),
        'Stopped By': res_biased.stopped_by,
    },
    {
        'Scenario': 'Fair coin (p=0.5)',
        'True Variance': var_fair,
        'Hoeffding Shots': hoeffding_shots,
        'EBS Shots': res_fair.n,
        'Savings': hoeffding_shots - res_fair.n,
        'Savings %': 100 * (1 - res_fair.n / hoeffding_shots),
        'Stopped By': res_fair.stopped_by,
    },
])

print("\n" + "="*80)
print("HOEFFDING vs EBS COMPARISON")
print("="*80)
print(comparison.to_string(index=False))
print("="*80)

## 8. Key Takeaways

### Hoeffding's Limitation
- Uses **worst-case variance** assumption ($\sigma^2 = R^2/4$)
- Plans a **fixed** number of shots, regardless of actual variance
- **Wasteful** when true variance is lower than worst-case

### EBS Advantages
- **Adapts** to observed variance during sampling
- **Stops early** when confidence threshold is met
- **Safe**: Never uses more shots than Hoeffding (has a cap)
- **Efficient**: Can save significant shots in low-variance scenarios

### When to Use Each

**Use Hoeffding when:**
- You need a **fixed budget** known in advance
- Simplicity is more important than efficiency
- You want guaranteed worst-case behavior

**Use EBS when:**
- You want to **minimize expected shots**
- Variance is unknown or expected to be low
- You're willing to accept **variable runtime**
- Cost savings matter (quantum hardware is expensive!)

### The Promise of EBS

> **EBS adapts to the data and never does worse than Hoeffding.**

In the next notebooks, we'll explore:
- The mathematical theory behind EBS (Notebook 06)
- Practical application to SWAP test fidelity (Notebook 07)
- Comprehensive Hoeffding vs EBS comparison (Notebook 08)
- Parameter tuning for optimal performance (Notebook 09)